# Garment Order Production Planning: Model Development and Comparison

This notebook is the complete, self-contained model development and comparison workflow for the garment order planning project. Feature engineering, candidate models, cross-validation, model selection, artifact export, and evaluation plots are defined inline; no project `.py` training modules are required.

The notebook reads the project’s supplied CSV files from `data/` and does not create or fill training records. Model outputs are estimates learned from the supplied records. Confirm that the files and recorded outcomes are suitable for the intended factory before using results for operational decisions.

Run cells from top to bottom. Trained model artifacts and evaluation reports are written to `models/`. The worker-operation model does not use worker IDs as model features; its output is a performance estimate, not a live roster or attendance check. Line and shift capacity is an aggregate plan and assumes the required staffing and machines are available.


In [10]:
# Resolve project paths without depending on the notebook's working directory.
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    parent_candidate = PROJECT_ROOT.parent
    if (parent_candidate / "data").is_dir():
        PROJECT_ROOT = parent_candidate

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "models"
REQUIRED_DATA_FILES = (
    "orders.csv",
    "allocations.csv",
    "interventions.csv",
    "workers.csv",
    "operations.csv",
)
missing_files = [name for name in REQUIRED_DATA_FILES if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Required source CSV files are missing from {DATA_DIR}: {', '.join(missing_files)}. "
        "Place the project data files in data/ before running this notebook."
    )

print(f"Project directory: {PROJECT_ROOT}")
print(f"Training data directory: {DATA_DIR}")
print(f"Model and report output directory: {OUTPUT_DIR}")

Project directory: d:\Freelance\Nadee\garment_order_system
Training data directory: d:\Freelance\Nadee\garment_order_system\data
Model and report output directory: d:\Freelance\Nadee\garment_order_system\models


## Data Contract, Targets, and Outputs

### Required source files

- `data/orders.csv`: order inputs, on-time outcome, staffing and shift fields, and completion outcome fields for capacity modelling.
- `data/allocations.csv`: completed worker-operation measurements used to build performance targets.
- `data/interventions.csv`: intervention inputs, selection outcome, and recorded cost target.
- `data/workers.csv`: grade, training, attendance, skill, and aggregate performance fields joined to completed allocations. Worker IDs are join keys only and are excluded from model features.
- `data/operations.csv`: operation requirements joined to allocation records.

The notebook checks required columns and key label conditions before fitting models. It does not generate or modify source rows. Feature functions apply explicit defaults to some missing worker attributes; review source completeness and those documented defaults before interpreting worker-operation results. If a required field or file is missing, it stops with an error for the data owner to resolve.

### Learning targets

| Analysis | Target | Validation |
| --- | --- | --- |
| N1 order outlook | `on_time_flag` | Stratified five-fold out-of-fold predictions |
| N2 intervention selection | `selected_flag` | Order-grouped five-fold out-of-fold predictions |
| Intervention cost | `total_cost` | Order-grouped five-fold out-of-fold predictions; MAE selection |
| N3 line-aware outlook | `on_time_flag` | Stratified five-fold out-of-fold predictions |
| N4 worker-operation performance | `actual_efficiency`, `actual_quality` | Order-grouped five-fold out-of-fold predictions |
| N5 line-shift capacity | Standard minutes per line-shift from recorded completions | Order-grouped out-of-fold predictions; MAE selection |

### Outputs

Selected model bundles, `metrics.json`, and `model_comparison.json` are saved under the project `models/` directory. The notebook’s comparison tables and plots are generated from out-of-fold results. Install compatible versions of NumPy, pandas, scikit-learn, TensorFlow, Joblib, Matplotlib, and IPython in the notebook environment before running all model candidates.

## Feature Engineering and Planning Inputs

In [11]:
"""Shared feature engineering used during training and live order prediction."""
SCENARIO_ENCODING = {
    name: index
    for index, name in enumerate(
        ["Current", "Extra_Workers", "Hybrid", "Machine_Adjust", "Outsourcing", "Overtime"]
    )
}
STYLE_ENCODING = {"Hoodie": 0, "Jacket": 1, "Legging": 2, "Polo": 3, "T-Shirt": 4}
PRIORITY_ENCODING = {"Low": 0, "Normal": 1, "High": 2}
STYLE_CAPACITY_FACTOR = {"Polo": 1.00, "T-Shirt": 1.02, "Hoodie": 0.94,
                         "Legging": 0.98, "Jacket": 0.92}
PRIORITY_CAPACITY_FACTOR = {"Low": 1.04, "Normal": 1.00, "High": 0.90}


def build_factory_profile(orders, allocations):
    """Estimate typical order and factory values from the available records."""
    per_order = allocations.groupby("order_id").agg(
        n_conflicts=("conflict_flag", "sum"),
        avg_worker_eff=("actual_efficiency", "mean"),
    )
    return {
        "smv": float(orders["required_smv_total"].median()),
        "unit_value": float((orders["order_value"] / orders["order_quantity"]).median()),
        "typical_workers": float(orders["n_available_workers"].median()),
        "typical_shifts": float(orders["n_shifts"].median()),
        "shift_hours": float(orders["shift_hours"].median()),
        "capacity_efficiency": float(orders["avg_worker_efficiency"].median()),
        "n_conflicts": float(per_order["n_conflicts"].median()),
        "avg_worker_eff": float(per_order["avg_worker_eff"].median()),
    }


def build_order_features(quantity, handover_days, smv, order_value, style, priority, profile):
    """Build matching features from quantity, days, SMV, value, style and priority."""
    quantity = float(quantity)
    handover_days = max(float(handover_days), 1.0)
    smv = float(smv)
    order_value = float(order_value)
    workload = quantity * smv
    daily_capacity = (
        profile["typical_workers"]
        * profile["shift_hours"]
        * 60
        * profile["capacity_efficiency"] / 100
        * profile["typical_shifts"]
    )
    return {
        "order_quantity": quantity,
        "handover_days": handover_days,
        "required_smv_total": smv,
        "order_value": order_value,
        "style_enc": STYLE_ENCODING[style],
        "priority_enc": PRIORITY_ENCODING[priority],
        "unit_value": order_value / quantity,
        "estimated_workload_min": workload,
        "estimated_workload_per_day": workload / handover_days,
        "capacity_ratio": (
            daily_capacity * handover_days / max(workload, 1)
            * STYLE_CAPACITY_FACTOR[style]
            * PRIORITY_CAPACITY_FACTOR[priority]
        ),
    }


def make_intervention_features(order_features, scenario, profile):
    """Append the factory-level and scenario features used by the N2 model."""
    row = {
        "scenario_enc": SCENARIO_ENCODING[scenario["scenario_type"]],
        "added_workers": float(scenario["added_workers"]),
        "overtime_hours": float(scenario["overtime_hours"]),
        "added_machines": float(scenario["added_machines"]),
        "outsourced_qty": float(scenario["outsourced_qty"]),
        "order_quantity": order_features["order_quantity"],
        "handover_days": order_features["handover_days"],
        "order_value": order_features["order_value"],
        "required_workload_min": order_features["estimated_workload_min"],
        "style_enc": order_features["style_enc"],
        "priority_enc": order_features["priority_enc"],
        "capacity_ratio": order_features["capacity_ratio"],
        "n_conflicts": profile["n_conflicts"],
        "avg_worker_eff": profile["avg_worker_eff"],
    }
    return row

import numpy as np

# Line-aware feasibility features shared with the live order planner.
LINE_GRADE_MIX = {"Jumper": 5, "A+": 4, "A": 4, "B": 3, "C": 1}
LINE_OPERATORS = sum(LINE_GRADE_MIX.values())
LINE_EFFICIENCY_TARGET = 0.70
LINE_SHIFT_HOURS = 8
LINE_MODEL_FEATURES = [
    "order_quantity", "handover_days", "required_smv_total", "order_value",
    "style_enc", "priority_enc", "assigned_lines", "shifts_per_day",
    "line_capacity_coverage", "unit_value",
]


def make_line_features(quantity, days, smv, order_value, style, priority, lines, shifts,
                       shift_hours=LINE_SHIFT_HOURS):
    workload = max(float(quantity) * float(smv), 1.0)
    daily_capacity = (
        max(int(lines), 1) * LINE_OPERATORS * float(shift_hours) * 60
        * LINE_EFFICIENCY_TARGET * max(int(shifts), 1)
    )
    coverage = (
        daily_capacity * max(int(days), 1) / workload
        * STYLE_CAPACITY_FACTOR[style] * PRIORITY_CAPACITY_FACTOR[priority]
    )
    return {
        "order_quantity": float(quantity),
        "handover_days": float(days),
        "required_smv_total": float(smv),
        "order_value": float(order_value),
        "style_enc": float(STYLE_ENCODING[style]),
        "priority_enc": float(PRIORITY_ENCODING[priority]),
        "assigned_lines": float(lines),
        "shifts_per_day": float(shifts),
        "line_capacity_coverage": float(coverage),
        "unit_value": float(order_value) / max(float(quantity), 1.0),
    }


def historical_line_count(worker_count):
    """Map historical headcount to full 17-operator line equivalents."""
    return max(1, min(30, int(np.floor(float(worker_count) / LINE_OPERATORS))))

WORKER_OPERATION_FEATURES = [
    "grade_rank", "experience_months", "is_trained", "attendance_rate",
    "worker_efficiency", "worker_quality", "worker_rework_rate",
    "previous_output", "operation_smv", "operation_sequence",
    "operation_complexity", "operation_quality_requirement",
    "operation_category", "machine_type", "skill_count", "skill_match",
]
GRADE_RANK = {"C": 0, "B": 1, "A": 2, "A+": 3, "Jumper": 4}
CATEGORY_RANK = {"Basic": 0, "Intermediate": 1, "Critical": 2}
MACHINE_TYPES = {name: index for index, name in enumerate(
    ["Bartack", "Buttonhole", "Flatlock", "Kansai", "Overlock", "SNLS"]
)}


def make_worker_operation_features(worker, operation):
    skills = {
        skill.strip().casefold()
        for skill in str(worker.get("machine_skills", "")).split(",")
        if skill.strip()
    }
    required_machine = str(operation.get("machine_type_required", "")).strip()
    return {
        "grade_rank": float(GRADE_RANK.get(str(worker.get("grade", "C")), 0)),
        "experience_months": float(worker.get("experience_months", 0) or 0),
        "is_trained": float(str(worker.get("training_status", "")).casefold() == "trained"),
        "attendance_rate": float(worker.get("attendance_rate", 0) or 0),
        "worker_efficiency": float(worker.get("efficiency_avg", 70) or 70),
        "worker_quality": float(worker.get("quality_avg", 85) or 85),
        "worker_rework_rate": float(worker.get("rework_rate", 0) or 0),
        "previous_output": float(worker.get("previous_output", 0) or 0),
        "operation_smv": float(operation.get("smv", 0) or 0),
        "operation_sequence": float(operation.get("sequence_no", 0) or 0),
        "operation_complexity": float(operation.get("complexity_score", 0) or 0),
        "operation_quality_requirement": float(operation.get("quality_requirement", 0) or 0),
        "operation_category": float(CATEGORY_RANK.get(str(operation.get("operation_category", "Basic")), 0)),
        "machine_type": float(MACHINE_TYPES.get(required_machine, -1)),
        "skill_count": float(len(skills)),
        "skill_match": float(required_machine.casefold() in skills),
    }

# N5 capacity features and recorded-output target.
POSTER_OPERATORS_PER_LINE = 17
REFERENCE_SHIFT_HOURS = 8.0
CAPACITY_MODEL_FEATURES = [
    "order_quantity", "handover_days", "required_smv_total", "style_enc",
    "assigned_lines", "shifts_per_day", "shift_hours",
]


def make_capacity_features(quantity, handover_days, smv, order_value,
                           style, priority, lines, shifts, shift_hours):
    quantity = float(quantity)
    return {
        "order_quantity": quantity,
        "handover_days": float(handover_days),
        "required_smv_total": float(smv),
        "style_enc": float(STYLE_ENCODING[style]),
        "assigned_lines": float(lines),
        "shifts_per_day": float(shifts),
        "shift_hours": float(shift_hours),
    }


def build_capacity_training_data(orders):
    """Build an order-level target from recorded output and completion dates."""
    data = orders.copy()
    start = pd.to_datetime(data["order_date"], errors="coerce")
    finish = pd.to_datetime(data["actual_completion_date"], errors="coerce")
    elapsed = (finish - start).dt.days.clip(lower=1)
    numeric_columns = [
        "order_quantity", "handover_days", "required_smv_total",
        "actual_completed_qty", "n_available_workers", "n_shifts", "shift_hours",
    ]
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    data["elapsed_days"] = elapsed
    data["assigned_lines"] = data["n_available_workers"] / POSTER_OPERATORS_PER_LINE
    data["shifts_per_day"] = data["n_shifts"]
    data["target_standard_minutes_per_line_shift"] = (
        data["actual_completed_qty"] * data["required_smv_total"] /
        data["elapsed_days"] / data["assigned_lines"] /
        data["shifts_per_day"] *
        (REFERENCE_SHIFT_HOURS / data["shift_hours"])
    )
    valid = (
        data["actual_completed_qty"].gt(0) &
        data["required_smv_total"].gt(0) &
        data["assigned_lines"].gt(0) &
        data["shifts_per_day"].gt(0) &
        data["shift_hours"].gt(0) &
        data["target_standard_minutes_per_line_shift"].gt(0) &
        data["style_id"].isin(STYLE_ENCODING)
    )
    data = data.loc[valid].replace([np.inf, -np.inf], np.nan)
    data = data.dropna(subset=CAPACITY_MODEL_FEATURES + [
        "target_standard_minutes_per_line_shift", "order_id",
    ])
    X = pd.DataFrame(
        [make_capacity_features(
            row.order_quantity, row.handover_days, row.required_smv_total,
            row.order_value, row.style_id, row.priority, row.assigned_lines,
            row.shifts_per_day, row.shift_hours,
        ) for row in data.itertuples()],
        columns=CAPACITY_MODEL_FEATURES,
    )
    y = data["target_standard_minutes_per_line_shift"].to_numpy(dtype=float)
    return X, y, data["order_id"].to_numpy(), data["style_id"].to_numpy()


## Neural Model Implementations

In [12]:
"""Scikit-learn-compatible neural models used in model comparison.

The graph classifier builds a feature-similarity graph over training examples.
Each row is a node; edges connect the nearest training rows, and a mean
neighbour message is concatenated with each node's own features.
"""
import numpy as np
import tensorflow as tf
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


class TensorFlowDNNClassifier(ClassifierMixin, BaseEstimator):
    """Dense neural classifier implemented with TensorFlow's Keras API."""

    def __init__(self, hidden_layer_sizes=(128, 64, 32), dropout=0.25,
                 learning_rate=0.001, epochs=100, batch_size=128, patience=8,
                 random_state=42):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.patience = patience
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        self.classes_ = np.unique(y.astype(int))
        self.scaler_ = StandardScaler().fit(X)
        X_scaled = self.scaler_.transform(X).astype(np.float32)

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(self.random_state)
        layers = [tf.keras.layers.Input(shape=(X_scaled.shape[1],))]
        for width in self.hidden_layer_sizes:
            layers.append(tf.keras.layers.Dense(width, activation="relu"))
            if self.dropout:
                layers.append(tf.keras.layers.Dropout(self.dropout))
        layers.append(tf.keras.layers.Dense(1, activation="sigmoid"))
        self.keras_model_ = tf.keras.Sequential(layers)
        self.keras_model_.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=self.patience, restore_best_weights=True
        )
        self.history_ = self.keras_model_.fit(
            X_scaled, y,
            validation_split=0.15,
            epochs=self.epochs,
            batch_size=self.batch_size,
            callbacks=[early_stopping],
            verbose=0,
        ).history
        return self

    def predict_proba(self, X):
        X_scaled = self.scaler_.transform(np.asarray(X, dtype=np.float32))
        positive = self.keras_model_.predict(X_scaled, verbose=0).reshape(-1)
        return np.column_stack([1.0 - positive, positive])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class GANNClassifier(ClassifierMixin, BaseEstimator):
    """Genetic-algorithm search over MLP architectures and learning settings."""

    def __init__(self, population_size=4, generations=2, max_iter=140,
                 random_state=42):
        self.population_size = population_size
        self.generations = generations
        self.max_iter = max_iter
        self.random_state = random_state

    def _model(self, chromosome, seed):
        hidden, alpha, learning_rate = chromosome
        return MLPClassifier(
            hidden_layer_sizes=hidden,
            alpha=alpha,
            learning_rate_init=learning_rate,
            max_iter=self.max_iter,
            early_stopping=True,
            validation_fraction=0.12,
            n_iter_no_change=12,
            random_state=seed,
        )

    @staticmethod
    def _score(model, X, y):
        probabilities = model.predict_proba(X)[:, 1]
        return brier_score_loss(y, probabilities)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        self.classes_ = np.unique(y)
        rng = np.random.RandomState(self.random_state)
        x_fit, x_valid, y_fit, y_valid = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state, stratify=y
        )

        hidden_choices = [(64, 32), (96, 48), (64, 32, 16), (128, 64)]
        alpha_choices = [0.0001, 0.001, 0.01]
        learning_rate_choices = [0.001, 0.003, 0.01]

        def random_chromosome():
            return (
                hidden_choices[rng.randint(len(hidden_choices))],
                alpha_choices[rng.randint(len(alpha_choices))],
                learning_rate_choices[rng.randint(len(learning_rate_choices))],
            )

        population = [random_chromosome() for _ in range(self.population_size)]
        fitness = {}
        for generation in range(self.generations):
            for chromosome in population:
                if chromosome not in fitness:
                    candidate = self._model(chromosome, self.random_state + generation)
                    candidate.fit(x_fit, y_fit)
                    fitness[chromosome] = self._score(candidate, x_valid, y_valid)
            ranked = sorted(population, key=lambda gene: fitness[gene])
            if generation + 1 < self.generations:
                parents = ranked[:max(2, len(ranked) // 2)]
                next_population = parents[:]
                while len(next_population) < self.population_size:
                    first = parents[rng.randint(len(parents))]
                    second = parents[rng.randint(len(parents))]
                    child = [first[0], second[1], first[2]]
                    if rng.rand() < 0.4:
                        child[0] = hidden_choices[rng.randint(len(hidden_choices))]
                    if rng.rand() < 0.4:
                        child[1] = alpha_choices[rng.randint(len(alpha_choices))]
                    if rng.rand() < 0.4:
                        child[2] = learning_rate_choices[rng.randint(len(learning_rate_choices))]
                    next_population.append(tuple(child))
                population = next_population

        self.best_config_ = min(fitness, key=fitness.get)
        self.model_ = self._model(self.best_config_, self.random_state)
        self.model_.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(np.asarray(X, dtype=float))

    def predict(self, X):
        return self.model_.predict(np.asarray(X, dtype=float))


class GraphNeuralNetworkClassifier(ClassifierMixin, BaseEstimator):
    """One-step graph message-passing classifier over feature-similar rows."""

    def __init__(self, n_neighbors=8, hidden_layer_sizes=(64, 32), max_iter=220,
                 random_state=42):
        self.n_neighbors = n_neighbors
        self.hidden_layer_sizes = hidden_layer_sizes
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        self.classes_ = np.unique(y)
        self.scaler_ = StandardScaler().fit(X)
        self.X_train_ = self.scaler_.transform(X)
        count = min(self.n_neighbors + 1, len(self.X_train_))
        self.neighbors_ = NearestNeighbors(n_neighbors=count).fit(self.X_train_)
        indices = self.neighbors_.kneighbors(self.X_train_, return_distance=False)

        aggregate = np.zeros_like(self.X_train_)
        for row_index, row_neighbors in enumerate(indices):
            # Self-loops are excluded; the remaining neighbours send messages.
            selected = [idx for idx in row_neighbors if idx != row_index][:self.n_neighbors]
            if selected:
                aggregate[row_index] = self.X_train_[selected].mean(axis=0)
        graph_features = np.hstack([self.X_train_, aggregate])
        self.model_ = MLPClassifier(
            hidden_layer_sizes=self.hidden_layer_sizes,
            alpha=0.01,
            max_iter=self.max_iter,
            early_stopping=True,
            validation_fraction=0.12,
            random_state=self.random_state,
        )
        self.model_.fit(graph_features, y)
        return self

    def _features(self, X):
        X_scaled = self.scaler_.transform(np.asarray(X, dtype=float))
        indices = self.neighbors_.kneighbors(
            X_scaled, n_neighbors=min(self.n_neighbors, len(self.X_train_)),
            return_distance=False,
        )
        aggregate = self.X_train_[indices].mean(axis=1)
        return np.hstack([X_scaled, aggregate])

    def predict_proba(self, X):
        return self.model_.predict_proba(self._features(X))

    def predict(self, X):
        return self.model_.predict(self._features(X))


## Train and Compare the Models

The notebook compares five planning tasks. N1 estimates order on-time outcomes from order inputs. N2 estimates which recorded intervention type is selected; this score is not a probability that an intervention improves delivery. The cost model estimates intervention cost. N3 estimates on-time feasibility using order and line-capacity inputs. N4 estimates worker-operation efficiency and quality without using worker IDs as model features. N5 estimates aggregate completed output per standardized line-shift.

Classification candidates are compared using out-of-fold Brier score and ROC-AUC. Intervention models and regressors use order-grouped folds where several rows belong to one order, reducing leakage between an order’s scenarios. Capacity targets are calculated from recorded completion quantities, elapsed time, staffing, shift count, shift duration, and SMV. The factory poster’s 17-operator grade mix is used to express historical headcount as line equivalents; it is a planning assumption, not a live availability record.


In [14]:
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    AdaBoostClassifier,
    AdaBoostRegressor,
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    brier_score_loss,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


SEED = 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=UserWarning)

orders = pd.read_csv(DATA_DIR / "orders.csv")
allocations = pd.read_csv(DATA_DIR / "allocations.csv")
scenarios = pd.read_csv(DATA_DIR / "interventions.csv")
workers = pd.read_csv(DATA_DIR / "workers.csv")
operations = pd.read_csv(DATA_DIR / "operations.csv")
# Validate the expected source fields before feature construction. Missing fields
# stop the workflow; the notebook does not invent replacement observations.
required_columns = {
    "orders": {
        "order_id", "order_date", "actual_completion_date", "order_quantity",
        "handover_days", "order_value", "required_smv_total", "style_id",
        "priority", "n_available_workers", "n_shifts", "shift_hours",
        "avg_worker_efficiency", "actual_completed_qty", "on_time_flag",
    },
    "allocations": {
        "order_id", "worker_id", "operation_id", "allocation_status",
        "conflict_flag", "actual_efficiency", "actual_quality",
    },
    "interventions": {
        "order_id", "scenario_type", "added_workers", "overtime_hours",
        "added_machines", "outsourced_qty", "total_cost", "selected_flag",
    },
    "workers": {
        "worker_id", "grade", "experience_months", "training_status",
        "attendance_rate", "machine_skills", "efficiency_avg", "quality_avg",
        "rework_rate", "previous_output",
    },
    "operations": {
        "operation_id", "operation_category", "machine_type_required", "smv",
        "sequence_no", "complexity_score", "quality_requirement",
    },
}
source_frames = {
    "orders": orders,
    "allocations": allocations,
    "interventions": scenarios,
    "workers": workers,
    "operations": operations,
}
for table_name, required in required_columns.items():
    missing = sorted(required.difference(source_frames[table_name].columns))
    if missing:
        raise ValueError(f"{table_name}.csv is missing required columns: {missing}")
    print(f"{table_name}: {len(source_frames[table_name]):,} rows")

if orders["order_id"].duplicated().any():
    raise ValueError("orders.csv must contain one row per order_id.")
if orders["on_time_flag"].nunique(dropna=True) != 2:
    raise ValueError("on_time_flag must contain both outcome classes for classification.")
if scenarios["selected_flag"].nunique(dropna=True) != 2:
    raise ValueError("selected_flag must contain both outcome classes for classification.")
if orders["on_time_flag"].value_counts().min() < 5:
    raise ValueError("Five-fold stratified validation needs at least five orders in each on-time class.")
if scenarios["order_id"].nunique() < 5:
    raise ValueError("Order-grouped validation needs at least five distinct intervention orders.")

profile = build_factory_profile(orders, allocations)

N1_FEATURES = [
    "order_quantity", "handover_days", "required_smv_total", "order_value",
    "style_enc", "priority_enc", "unit_value", "estimated_workload_min",
    "estimated_workload_per_day", "capacity_ratio",
]


def classification_metrics(y_true, probabilities):
    predictions = (probabilities >= 0.5).astype(int)
    return {
        "accuracy": round(float(accuracy_score(y_true, predictions)), 4),
        "precision": round(float(precision_score(y_true, predictions, zero_division=0)), 4),
        "recall": round(float(recall_score(y_true, predictions, zero_division=0)), 4),
        "f1": round(float(f1_score(y_true, predictions, zero_division=0)), 4),
        "confusion_matrix": confusion_matrix(y_true, predictions).tolist(),
        "roc_auc": round(float(roc_auc_score(y_true, probabilities)), 4),
        "brier_score": round(float(brier_score_loss(y_true, probabilities)), 4),
    }


def compare_classifiers(name, candidates, X, y, cv, groups=None):
    scores = {}
    for model_name, model in candidates.items():
        print(f"  Training {model_name}...", flush=True)
        kwargs = {"groups": groups} if groups is not None else {}
        probabilities = cross_val_predict(
            model, X, y, cv=cv, method="predict_proba", n_jobs=1, **kwargs
        )[:, 1]
        scores[model_name] = classification_metrics(y, probabilities)
    # Brier score rewards useful, well-calibrated probabilities; AUC breaks ties.
    winner = min(scores, key=lambda key: (scores[key]["brier_score"], -scores[key]["roc_auc"]))
    final_model = candidates[winner].fit(X, y)
    scores = dict(sorted(
        scores.items(), key=lambda item: (item[1]["brier_score"], -item[1]["roc_auc"])
    ))
    print(f"\n{name} comparison (sorted by Brier score, then ROC-AUC):")
    for model_name, metrics in scores.items():
        marker = "  <-- BEST" if model_name == winner else ""
        print(
            f"  {model_name:<30} Brier={metrics['brier_score']:.4f}  "
            f"ROC-AUC={metrics['roc_auc']:.4f}  F1={metrics['f1']:.4f}{marker}"
        )
    return winner, final_model, scores


def save_classifier_bundle(name, model, features, profile, output_path,
                           keras_path, extra=None):
    """Save sklearn models in joblib and Keras networks in native .keras format."""
    bundle = {
        "name": name,
        "features": features,
        "profile": profile,
        "training_signature": "six-order-inputs-v1",
    }
    if extra:
        bundle.update(extra)
    if isinstance(model, TensorFlowDNNClassifier):
        model.keras_model_.save(keras_path)
        bundle.update({
            "model": None,
            "type": "keras",
            "model_path": str(Path(keras_path).relative_to(PROJECT_ROOT)),
            "scaler": model.scaler_,
        })
    else:
        bundle.update({"model": model, "type": "sklearn"})
    joblib.dump(bundle, output_path)


# N1: use the six order details available in the form, plus workload and
# capacity features derived from those inputs and the factory profile.
X1 = pd.DataFrame(
    [
        build_order_features(
            row.order_quantity, row.handover_days, row.required_smv_total,
            row.order_value, row.style_id, row.priority, profile,
        )
        for row in orders.itertuples()
    ],
    columns=N1_FEATURES,
)
y1 = orders["on_time_flag"].astype(int).to_numpy()

n1_candidates = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=160, max_depth=10, min_samples_leaf=8,
        n_jobs=-1, random_state=SEED,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=160, max_depth=12, min_samples_leaf=6,
        n_jobs=-1, random_state=SEED,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=12, class_weight="balanced", random_state=SEED,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=2, learning_rate=0.06, random_state=SEED,
    ),
    "Histogram Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=140, max_leaf_nodes=15, l2_regularization=2.0,
        learning_rate=0.07, random_state=SEED,
    ),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, learning_rate=0.05, random_state=SEED),
    "K-Nearest Neighbors": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1)
    ),
    "DNN (TensorFlow/Keras)": TensorFlowDNNClassifier(random_state=SEED),
    "Genetic Algorithm Neural Network (GANN)": make_pipeline(
        StandardScaler(), GANNClassifier(population_size=4, generations=2,
                                         max_iter=100, random_state=SEED)
    ),
    "Graph Neural Network (GNN)": GraphNeuralNetworkClassifier(random_state=SEED),
    "Gaussian Naive Bayes": make_pipeline(StandardScaler(), GaussianNB()),
}
n1_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
n1_name, n1_model, n1_metrics = compare_classifiers(
    "On-time model", n1_candidates, X1, y1, n1_cv
)

# Dedicated fit for the Keras training-curve visualization below.
dnn_history_model = TensorFlowDNNClassifier(random_state=SEED).fit(X1, y1)
save_classifier_bundle(
    n1_name, n1_model, N1_FEATURES, profile,
    (OUTPUT_DIR / "best_n1_model.pkl"), (OUTPUT_DIR / "best_n1_model.keras"),
    extra={"user_inputs": ["order_quantity", "handover_days", "required_smv_total", "order_value", "style_id", "priority"]},
)

# N2: preserve scenario labels and costs, but build the order-level fields with
# the same inferred profile used at prediction time. Grouped folds keep all six
# scenarios for an order on the same side of each validation split.
order_features = {
    int(row.order_id): build_order_features(
        row.order_quantity, row.handover_days, row.required_smv_total,
        row.order_value, row.style_id, row.priority, profile,
    )
    for row in orders.itertuples()
}
n2_feature_rows = []
for scenario in scenarios.to_dict(orient="records"):
    inferred = order_features[int(scenario["order_id"])]
    n2_feature_rows.append(make_intervention_features(inferred, scenario, profile))

N2_FEATURES = [
    "scenario_enc", "added_workers", "overtime_hours", "added_machines",
    "outsourced_qty", "order_quantity", "handover_days", "order_value",
    "required_workload_min", "style_enc", "priority_enc", "capacity_ratio",
    "n_conflicts", "avg_worker_eff",
]
X2 = pd.DataFrame(n2_feature_rows, columns=N2_FEATURES)
y2 = scenarios["selected_flag"].astype(int).to_numpy()
groups = scenarios["order_id"].to_numpy()
n2_cv = GroupKFold(n_splits=5)
n2_candidates = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=120, max_depth=14, min_samples_leaf=5,
        n_jobs=-1, random_state=SEED,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=140, max_depth=16, min_samples_leaf=4,
        n_jobs=-1, random_state=SEED,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=10, class_weight="balanced", random_state=SEED,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=2, learning_rate=0.06, random_state=SEED,
    ),
    "Histogram Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=140, max_leaf_nodes=19, l2_regularization=2.0,
        learning_rate=0.07, random_state=SEED,
    ),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, learning_rate=0.05, random_state=SEED),
    "K-Nearest Neighbors": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=25, weights="distance", n_jobs=-1)
    ),
    "DNN (TensorFlow/Keras)": TensorFlowDNNClassifier(random_state=SEED),
    "Genetic Algorithm Neural Network (GANN)": make_pipeline(
        StandardScaler(), GANNClassifier(population_size=4, generations=2,
                                         max_iter=100, random_state=SEED)
    ),
    "Graph Neural Network (GNN)": GraphNeuralNetworkClassifier(random_state=SEED),
    "Gaussian Naive Bayes": make_pipeline(StandardScaler(), GaussianNB()),
}
n2_name, n2_model, n2_metrics = compare_classifiers(
    "Intervention model", n2_candidates, X2, y2, n2_cv, groups
)
save_classifier_bundle(
    n2_name, n2_model, N2_FEATURES, profile,
    (OUTPUT_DIR / "best_n2_model.pkl"), (OUTPUT_DIR / "best_n2_model.keras"),
    extra={"scenario_encoding": SCENARIO_ENCODING},
)

# The cost regressor is trained and evaluated with the same order-grouped folds.
cost_candidates = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=12, min_samples_leaf=8, random_state=SEED,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=120, max_depth=16, min_samples_leaf=4,
        n_jobs=-1, random_state=SEED,
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=120, max_depth=16, min_samples_leaf=4,
        n_jobs=-1, random_state=SEED,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, max_depth=2, learning_rate=0.06, loss="huber", random_state=SEED,
    ),
    "Histogram Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=160, max_leaf_nodes=19, l2_regularization=2.0,
        learning_rate=0.07, random_state=SEED,
    ),
    "AdaBoost": AdaBoostRegressor(n_estimators=100, learning_rate=0.05, random_state=SEED),
    "K-Nearest Neighbors": make_pipeline(
        StandardScaler(), KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=-1)
    ),
}
cost_scores = {}
cost_models = {}
for model_name, model in cost_candidates.items():
    print(f"  Training cost model {model_name}...", flush=True)
    oof_cost = cross_val_predict(model, X2, scenarios["total_cost"].to_numpy(),
                                 cv=n2_cv, groups=groups, n_jobs=1)
    cost_scores[model_name] = {
        "mae": round(float(mean_absolute_error(scenarios["total_cost"], oof_cost)), 2),
        "r2": round(float(r2_score(scenarios["total_cost"], oof_cost)), 4),
    }
    cost_models[model_name] = model
cost_name = min(cost_scores, key=lambda key: cost_scores[key]["mae"])
cost_scores = dict(sorted(cost_scores.items(), key=lambda item: item[1]["mae"]))
print("\nCost model comparison (sorted by MAE):")
for model_name, metrics in cost_scores.items():
    marker = "  <-- BEST" if model_name == cost_name else ""
    print(f"  {model_name:<30} MAE={metrics['mae']:.2f}  R²={metrics['r2']:.4f}{marker}")
cost_model = cost_models[cost_name].fit(X2, scenarios["total_cost"].to_numpy())
joblib.dump(cost_model, (OUTPUT_DIR / "best_cost_model.pkl"))

# N3: compare order feasibility models with line and shift capacity inputs.
# Historical headcount is converted to 17-operator line equivalents using the
# factory poster's staffing standard. This is an input feature, not live roster data.
line_feature_rows = []
for row in orders.itertuples():
    line_count = historical_line_count(row.n_available_workers)
    line_feature_rows.append(make_line_features(
        row.order_quantity, row.handover_days, row.required_smv_total,
        row.order_value, row.style_id, row.priority, line_count, row.n_shifts,
        row.shift_hours,
    ))
X3 = pd.DataFrame(line_feature_rows, columns=LINE_MODEL_FEATURES)
y3 = orders["on_time_flag"].astype(int).to_numpy()
line_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
line_candidates = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(C=0.8, max_iter=2000, random_state=SEED)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=160, max_depth=10, min_samples_leaf=8,
        n_jobs=-1, random_state=SEED,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=160, max_depth=12, min_samples_leaf=6,
        n_jobs=-1, random_state=SEED,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=2, learning_rate=0.06, random_state=SEED,
    ),
    "Histogram Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=140, max_leaf_nodes=15, l2_regularization=2.0,
        learning_rate=0.07, random_state=SEED,
    ),
}
line_model_name, line_model, line_metrics = compare_classifiers(
    "Line-aware feasibility", line_candidates, X3, y3, line_cv
)
line_model_metrics = line_metrics[line_model_name]
joblib.dump({
    "name": line_model_name,
    "model": line_model,
    "features": LINE_MODEL_FEATURES,
}, (OUTPUT_DIR / "line_feasibility_model.pkl"))
# N4: predict operator efficiency and quality for each required operation.
performance_rows = allocations.merge(workers, on="worker_id").merge(operations, on="operation_id")
performance_rows = performance_rows[
    performance_rows["allocation_status"].eq("Completed")
].dropna(subset=["actual_efficiency", "actual_quality"])
X4 = pd.DataFrame(
    [make_worker_operation_features(row, row) for row in performance_rows.to_dict(orient="records")],
    columns=WORKER_OPERATION_FEATURES,
)
y4 = performance_rows[["actual_efficiency", "actual_quality"]].astype(float).to_numpy()
workforce_candidates = {
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100, max_depth=16, min_samples_leaf=8,
        n_jobs=-1, random_state=SEED,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=16, min_samples_leaf=8,
        n_jobs=-1, random_state=SEED,
    ),
}
if performance_rows["order_id"].nunique() < 5:
    raise ValueError("Worker-operation validation needs at least five distinct orders.")
performance_cv = GroupKFold(n_splits=5)
workforce_scores = {}
workforce_groups = performance_rows["order_id"].to_numpy()
for model_name, candidate in workforce_candidates.items():
    workforce_oof = cross_val_predict(
        candidate, X4, y4, cv=performance_cv,
        groups=workforce_groups, n_jobs=1,
    )
    workforce_scores[model_name] = {
        "efficiency_mae": round(float(mean_absolute_error(y4[:, 0], workforce_oof[:, 0])), 3),
        "efficiency_r2": round(float(r2_score(y4[:, 0], workforce_oof[:, 0])), 4),
        "quality_mae": round(float(mean_absolute_error(y4[:, 1], workforce_oof[:, 1])), 3),
        "quality_r2": round(float(r2_score(y4[:, 1], workforce_oof[:, 1])), 4),
    }
workforce_best = min(
    workforce_scores,
    key=lambda name: (workforce_scores[name]["efficiency_mae"] + workforce_scores[name]["quality_mae"]) / 2,
)
performance_metrics = workforce_scores[workforce_best]
performance_model = workforce_candidates[workforce_best]
performance_model.fit(X4, y4)
joblib.dump({
    "name": f"Worker-operation {workforce_best} Regressor",
    "model": performance_model,
    "features": WORKER_OPERATION_FEATURES,
}, (OUTPUT_DIR / "workforce_performance_model.pkl"))
print(
    f"\nWorker-operation model validation ({workforce_best}): "
    f"Efficiency MAE={performance_metrics['efficiency_mae']:.3f}, "
    f"Quality MAE={performance_metrics['quality_mae']:.3f}"
)

# N5: train aggregate line-shift capacity from recorded completed orders.
X5, y5, capacity_groups, capacity_styles = build_capacity_training_data(orders)
if len(X5) < 20 or len(np.unique(capacity_groups)) < 5:
    raise ValueError("Capacity model requires at least 20 valid completed orders across 5 distinct orders.")
capacity_cv = GroupKFold(n_splits=min(5, len(np.unique(capacity_groups))))
capacity_candidates = {
    "Extra Trees": ExtraTreesRegressor(n_estimators=180, max_depth=18, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
    "Random Forest": RandomForestRegressor(n_estimators=180, max_depth=16, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
    "Gradient Boosting (Huber)": GradientBoostingRegressor(n_estimators=140, max_depth=2, min_samples_leaf=8, learning_rate=0.05, loss="huber", random_state=SEED),
}
capacity_scores, capacity_oof = {}, {}
for model_name, candidate in capacity_candidates.items():
    print(f"  Training capacity model {model_name}...", flush=True)
    oof = cross_val_predict(candidate, X5, y5, cv=capacity_cv, groups=capacity_groups, n_jobs=1)
    capacity_oof[model_name] = oof
    capacity_scores[model_name] = {
        "mae_standard_minutes_per_line_shift": round(float(mean_absolute_error(y5, oof)), 3),
        "r2": round(float(r2_score(y5, oof)), 4),
    }
capacity_best = min(capacity_scores, key=lambda name: capacity_scores[name]["mae_standard_minutes_per_line_shift"])
capacity_model = capacity_candidates[capacity_best].fit(X5, y5)
residuals = y5 - capacity_oof[capacity_best]
residual_quantiles = np.quantile(residuals, [0.25, 0.5, 0.75])
style_bounds = {}
for style_name in sorted(set(capacity_styles)):
    values = y5[capacity_styles == style_name]
    style_bounds[style_name] = {"p05": float(np.quantile(values, 0.05)), "p95": float(np.quantile(values, 0.95)), "rows": int(len(values))}
capacity_bundle = {
    "name": capacity_best, "model": capacity_model, "features": list(X5.columns),
    "residual_quantiles": {"p25": float(residual_quantiles[0]), "p50": float(residual_quantiles[1]), "p75": float(residual_quantiles[2])},
    "style_target_bounds": style_bounds, "training_rows": int(len(X5)),
    "training_orders": int(len(np.unique(capacity_groups))),
    "target": "recorded completed quantity * SMV / elapsed days / historical line equivalents / shifts, normalized to 8 hours",
    "validation": "5-fold order-grouped out-of-fold MAE and R2",
}
joblib.dump(capacity_bundle, (OUTPUT_DIR / "line_shift_capacity_model.pkl"))
capacity_metrics = capacity_scores[capacity_best]
capacity_scores = dict(sorted(capacity_scores.items(), key=lambda item: item[1]["mae_standard_minutes_per_line_shift"]))
print(f"Line-shift capacity model: {capacity_best}; MAE={capacity_metrics['mae_standard_minutes_per_line_shift']:.3f}; R2={capacity_metrics['r2']:.4f}; rows={len(X5)}")
report = {
    "data_note": "Validation metrics use the supplied project records; confirm their source and outcome definitions before interpretation.",
    "training_orders": int(len(orders)),
    "N1_user_inputs": ["order_quantity", "handover_days", "required_smv_total", "order_value", "style_id", "priority"],
    "training_signature": "six-order-inputs-v1",
    "N1_features": N1_FEATURES,
    "N1_best": n1_name,
    "N1_selection_metric": "Lowest 5-fold cross-validated Brier score; ROC-AUC breaks ties",
    "N1_models": n1_metrics,
    "N2_best": n2_name,
    "N2_selection_metric": "Lowest order-grouped 5-fold cross-validated Brier score; ROC-AUC breaks ties",
    "N2_models": n2_metrics,
    "cost_best": cost_name,
    "cost_models": cost_scores,
    "line_model": line_model_name,
    "line_model_candidates": line_metrics,
    "line_model_metrics": line_model_metrics,
    "workforce_model": f"Worker-operation {workforce_best} Regressor",
    "workforce_model_metrics": performance_metrics,
    "workforce_models": workforce_scores,
    "workforce_training_allocations": int(len(performance_rows)),
    "workforce_training_orders": int(performance_rows["order_id"].nunique()),
    "capacity_model": capacity_best,
    "capacity_model_metrics": capacity_metrics,
    "capacity_model_candidates": capacity_scores,
    "capacity_model_training_rows": int(len(X5)),
    "capacity_model_training_orders": int(len(np.unique(capacity_groups))),
    "capacity_model_features": list(X5.columns),
    "capacity_model_target": capacity_bundle["target"],
}
with open(OUTPUT_DIR / "model_comparison.json", "w", encoding="utf-8") as report_file:
    json.dump(report, report_file, indent=2)
with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as metrics_file:
    json.dump({"N1": n1_metrics, "N2": n2_metrics, "cost": cost_scores,
               "line_feasibility": line_metrics,
               "workforce_performance": workforce_scores,
               "line_shift_capacity": capacity_scores}, metrics_file, indent=2)

print("Saved trained models and cross-validation metrics under the configured model output directory.")


orders: 3,000 rows
allocations: 19,382 rows
interventions: 18,000 rows
workers: 150 rows
operations: 12 rows
  Training Logistic Regression...
  Training Random Forest...
  Training Extra Trees...
  Training Decision Tree...
  Training Gradient Boosting...
  Training Histogram Gradient Boosting...
  Training AdaBoost...
  Training K-Nearest Neighbors...
  Training DNN (TensorFlow/Keras)...
  Training Genetic Algorithm Neural Network (GANN)...
  Training Graph Neural Network (GNN)...
  Training Gaussian Naive Bayes...

On-time model comparison (sorted by Brier score, then ROC-AUC):
  DNN (TensorFlow/Keras)         Brier=0.0875  ROC-AUC=0.8494  F1=0.9312  <-- BEST
  Extra Trees                    Brier=0.0878  ROC-AUC=0.8555  F1=0.9326
  Logistic Regression            Brier=0.0879  ROC-AUC=0.8545  F1=0.9332
  Graph Neural Network (GNN)     Brier=0.0882  ROC-AUC=0.8430  F1=0.9347
  Gradient Boosting              Brier=0.0883  ROC-AUC=0.8498  F1=0.9316
  Genetic Algorithm Neural Network (G

KeyError: ['style_enc']

## Evaluation Plots

In [ ]:
"""Compare validation results for order, intervention, line, workforce and line-shift capacity models."""
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "#fbfdff"})


def plot_classifier_comparison(title, scores, best_name):
    ranked = sorted(scores.items(), key=lambda item: item[1]["brier_score"])
    names = [name for name, _ in ranked]
    auc = [item[1]["roc_auc"] for item in ranked]
    brier = [item[1]["brier_score"] for item in ranked]
    colors = ["#e87932" if name == best_name else "#3978b8" for name in names]
    y = np.arange(len(names))
    fig, axes = plt.subplots(1, 2, figsize=(14, max(5, len(names) * 0.42)))
    for ax, values, label, better in [
        (axes[0], auc, "ROC-AUC (higher is better)", "Ranking ability"),
        (axes[1], brier, "Brier score (lower is better)", "Probability quality"),
    ]:
        ax.barh(y, values, color=colors)
        ax.set_yticks(y, labels=names)
        ax.invert_yaxis()
        ax.set_xlabel(label)
        ax.set_title(better)
        ax.grid(axis="x", alpha=0.2)
    fig.suptitle(f"{title} model comparison | Selected: {best_name}", fontweight="bold")
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def plot_confusions(title, scores):
    names = list(scores)
    columns = 3
    rows = int(np.ceil(len(names) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(14, rows * 3.5), squeeze=False)
    for ax, name in zip(axes.flat, names):
        matrix = np.asarray(scores[name]["confusion_matrix"], dtype=int)
        row_totals = matrix.sum(axis=1, keepdims=True)
        rates = np.divide(matrix, row_totals, out=np.zeros_like(matrix, dtype=float), where=row_totals != 0)
        ax.imshow(rates, cmap="Blues", vmin=0, vmax=1)
        for row in range(2):
            for column in range(2):
                ax.text(column, row, f"{matrix[row, column]}\n{rates[row, column]:.0%}", ha="center", va="center")
        ax.set_xticks([0, 1], labels=["Predicted 0", "Predicted 1"])
        ax.set_yticks([0, 1], labels=["Actual 0", "Actual 1"])
        ax.set_title(name, fontsize=9)
    for ax in axes.flat[len(names):]:
        ax.axis("off")
    fig.suptitle(f"{title}: out-of-fold confusion matrices")
    fig.tight_layout()
    display(fig)
    plt.close(fig)


plot_classifier_comparison("N1 on-time", report["N1_models"], report["N1_best"])
plot_classifier_comparison("N2 intervention selection", report["N2_models"], report["N2_best"])
plot_confusions("N1 on-time", report["N1_models"])
plot_confusions("N2 intervention selection", report["N2_models"])

cost_names = list(report["cost_models"])
cost_mae = [report["cost_models"][name]["mae"] / 1000 for name in cost_names]
cost_r2 = [report["cost_models"][name]["r2"] for name in cost_names]
cost_colors = ["#e87932" if name == report["cost_best"] else "#3978b8" for name in cost_names]
y = np.arange(len(cost_names))
fig, axes = plt.subplots(1, 2, figsize=(13, max(5, len(cost_names) * 0.48)))
axes[0].barh(y, cost_mae, color=cost_colors)
axes[0].set_yticks(y, labels=cost_names)
axes[0].invert_yaxis()
axes[0].set_xlabel("MAE (LKR thousands, lower is better)")
axes[0].set_title("Cost prediction error")
axes[0].grid(axis="x", alpha=0.2)
axes[1].barh(y, cost_r2, color=cost_colors)
axes[1].set_yticks(y, labels=cost_names)
axes[1].invert_yaxis()
axes[1].set_xlabel("R² (higher is better)")
axes[1].set_title("Variance explained")
axes[1].grid(axis="x", alpha=0.2)
fig.suptitle(f"Intervention cost model comparison | Selected: {report['cost_best']}")
fig.tight_layout()
display(fig)
plt.close(fig)

plot_classifier_comparison(
    "Line-aware feasibility", report["line_model_candidates"], report["line_model"]
)
workforce_names = list(report["workforce_models"])
efficiency_mae = [report["workforce_models"][name]["efficiency_mae"] for name in workforce_names]
quality_mae = [report["workforce_models"][name]["quality_mae"] for name in workforce_names]
x = np.arange(len(workforce_names))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, efficiency_mae, width, label="Efficiency MAE", color="#3978b8")
ax.bar(x + width / 2, quality_mae, width, label="Quality MAE", color="#e87932")
ax.set_xticks(x, labels=workforce_names)
ax.set_ylabel("Mean absolute error (percentage points; lower is better)")
ax.set_title(f"N4 worker-operation model comparison | Selected: {report['workforce_model']}")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
display(fig)
plt.close(fig)

capacity_names = list(report["capacity_model_candidates"])
capacity_mae = [report["capacity_model_candidates"][name]["mae_standard_minutes_per_line_shift"] for name in capacity_names]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(capacity_names, capacity_mae, color="#5b8e7d")
ax.set_ylabel("Out-of-fold MAE (standard minutes per line-shift; lower is better)")
ax.set_title(f"N5 line-shift capacity model | Selected: {report['capacity_model']}")
ax.tick_params(axis="x", rotation=12)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
display(fig)
plt.close(fig)
history = dnn_history_model.history_
epochs = np.arange(1, len(history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history["loss"], label="Training")
axes[0].plot(epochs, history["val_loss"], label="Validation")
axes[0].set_title("Keras DNN loss by epoch")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.2)
axes[1].plot(epochs, history["accuracy"], label="Training")
axes[1].plot(epochs, history["val_accuracy"], label="Validation")
axes[1].set_title("Keras DNN accuracy by epoch")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.2)
fig.tight_layout()
display(fig)
plt.close(fig)

## Model Rankings and Evaluation Summary

The tables and plots below report out-of-fold validation results for each candidate. They describe performance on the supplied project records under the stated fold design; they are not guarantees for future orders or evidence of current worker, line, shift, or machine availability.


In [ ]:
import pandas as pd
from IPython.display import display

print(f"Training orders: {report['training_orders']:,}")
print(f"Completed worker-operation allocations: {report['workforce_training_allocations']:,}")
print(f"Orders represented in workforce training: {report['workforce_training_orders']:,}")
print(f"N1 selected model: {report['N1_best']}")
print(f"N2 selected model: {report['N2_best']}")
print(f"Cost selected model: {report['cost_best']}")
print(f"Line feasibility model: {report['line_model']}")
print(f"Workforce model: {report['workforce_model']}")
print(f"Capacity model: {report['capacity_model']} | MAE={report['capacity_model_metrics']['mae_standard_minutes_per_line_shift']:.3f} | trained orders={report['capacity_model_training_orders']:,}")
print(report['data_note'])

n1_table = pd.DataFrame(report['N1_models']).T.drop(columns=['confusion_matrix'], errors='ignore').sort_values('brier_score')
n1_table.index.name = 'N1 model'
display(n1_table.style.format('{:.4f}').highlight_min(subset=['brier_score'], color='#ffe3cc'))

n2_table = pd.DataFrame(report['N2_models']).T.drop(columns=['confusion_matrix'], errors='ignore').sort_values('brier_score')
n2_table.index.name = 'N2 model'
display(n2_table.style.format('{:.4f}').highlight_min(subset=['brier_score'], color='#ffe3cc'))

cost_table = pd.DataFrame(report['cost_models']).T.sort_values('mae')
cost_table.index.name = 'Cost model'
display(cost_table.style.format({'mae': '{:,.2f}', 'r2': '{:.4f}'}).highlight_min(subset=['mae'], color='#ffe3cc'))

line_table = pd.DataFrame(report['line_model_candidates']).T.drop(columns=['confusion_matrix'], errors='ignore').sort_values('brier_score')
line_table.index.name = 'N3 line-aware feasibility model'
display(line_table.style.format('{:.4f}'))

workforce_table = pd.DataFrame(report['workforce_models']).T
workforce_table['mean_target_mae'] = (workforce_table['efficiency_mae'] + workforce_table['quality_mae']) / 2
workforce_table = workforce_table.sort_values('mean_target_mae').drop(columns=['mean_target_mae'])
workforce_table.index.name = 'N4 workforce model'
display(workforce_table.style.format('{:.3f}').highlight_min(subset=['efficiency_mae', 'quality_mae'], color='#ffe3cc'))
capacity_table = pd.DataFrame(report['capacity_model_candidates']).T.sort_values('mae_standard_minutes_per_line_shift')
capacity_table.index.name = 'N5 capacity model'
display(capacity_table.style.format({'mae_standard_minutes_per_line_shift': '{:.3f}', 'r2': '{:.4f}'}).highlight_min(subset=['mae_standard_minutes_per_line_shift'], color='#ffe3cc'))

## Interpretation and Operational Limits

- Review the source CSV definitions and confirm recorded outcomes before interpreting model scores.
- Cross-validation is used for model comparison and selection. Since the same folds guide selection, report the selected model's scores as comparative estimates; use a separate time-based or order-grouped holdout for final research claims when enough records are available.
- N2 ranks similarity to intervention choices in the supplied records. It does not estimate causal delivery improvement.
- Intervention cost predictions depend on the cost fields present in the supplied records and must be checked against current wage, overtime, machine, and supplier rates.
- Worker-operation predictions use grade and performance-related fields, not worker IDs. They do not provide named assignments, live attendance, or current availability.
- The line-shift capacity model predicts aggregate output under the input assumptions. It does not verify that the grade mix, machines, or shifts are available.
- The factory poster's 97% quality value is a target reference. This notebook does not present it as a predicted quality outcome.
- Keep the source CSV files unchanged and retain the generated model and metric artifacts with the experiment record.
